# SOFR SDR Seasonality Analysis

This notebook analyzes USD SOFR OIS Swaps and Swaptions from SDR data to identify:
- **Month-end flows**: What trades happen at month end?
- **Quarter-end flows**: What trades happen at quarter end?
- **FOMC-related flows**: What trades happen around FOMC meetings?
- **Package structures**: Curve trades, butterflies, straddles

## Trade Classification Logic
- **Curve trades**: Two legs with matching PV01 within 1 minute window
- **Butterfly trades**: Three legs where belly has ~2x PV01 of wings
- **Straddles/Strangles**: Call + Put pairs with same/different strikes

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
 'figure.figsize': (16, 9),
 'axes.labelsize': 'x-large',
 'axes.titlesize':'x-large',
 'xtick.labelsize':'x-large',
 'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [ ]:
# Import SDR Data Builder
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

# Import seasonality analysis module
from sofr_sdr_seasonality_analysis import (
    run_sofr_seasonality_analysis,
    filter_sofr_trades,
    filter_new_trades,
    classify_all_trades,
    classifications_to_dataframe,
    add_event_classifications,
    generate_seasonality_report,
    get_fomc_dates,
    get_month_end_dates,
    get_quarter_end_dates,
)

print("Imports complete!")

## 1. Load SDR Data

Load historical SDR data for analysis. We'll analyze several months of data to identify seasonality patterns.

In [ ]:
# Initialize SDR Data Builder
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics.cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

print("SDR Data Builder initialized")

In [ ]:
# Define analysis period
# Use several months to capture multiple month-ends, quarter-ends, and FOMC dates
start_date = datetime.date(2024, 9, 1)  # Start from September 2024
end_date = datetime.date(2024, 12, 20)  # Through December 2024

print(f"Analysis period: {start_date} to {end_date}")
print(f"This covers ~{(end_date - start_date).days} days")

In [ ]:
# Fetch historical SDR trades
print("Fetching SDR data...")

raw_df = sdr.grab_historical_sdr_trades(
    start_date=start_date,
    end_date=end_date,
    agency="CFTC",
    asset_class="RATES",
    one_df=True,
)

print(f"Loaded {len(raw_df):,} raw trades")
raw_df.head()

## 2. Filter and Classify Trades

Filter to SOFR trades and classify by:
- Product type (OIS swap, swaption, cap, floor)
- Tenor (1Y, 2Y, 5Y, 10Y, 30Y, etc.)
- Forward start (spot, 1Y, 5Y, etc.)
- Package type (outright, curve, fly, straddle)

In [ ]:
# Run the complete analysis pipeline
trades_df, report = run_sofr_seasonality_analysis(raw_df, verbose=True)

In [ ]:
# View sample of classified trades
print("Sample classified trades:")
display(trades_df.head(20))

In [ ]:
# Trade classification summary
print("\n=== TRADE CLASSIFICATION SUMMARY ===")
print(f"\nTotal trades analyzed: {len(trades_df):,}")

print("\n--- By Product Type ---")
print(trades_df["product_type"].value_counts())

print("\n--- By Package Type ---")
print(trades_df["package_type"].value_counts())

print("\n--- Top 20 Trade Labels by Count ---")
print(trades_df["trade_label"].value_counts().head(20))

## 3. Overall Flow Analysis

Analyze overall trading patterns before diving into seasonality.

In [ ]:
# Volume by trade label
print("\n=== TOP TRADES BY NOTIONAL VOLUME ===")
volume_by_label = report.get("overall_volume_by_label", pd.Series())
print(volume_by_label.head(25).to_string())

In [ ]:
# Plot top trades by volume
top_trades = volume_by_label.head(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(top_trades)), top_trades.values / 1e9)
ax.set_yticks(range(len(top_trades)))
ax.set_yticklabels(top_trades.index)
ax.set_xlabel("Total Notional Volume ($ Billions)")
ax.set_title("Top 15 SOFR Trades by Notional Volume")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Interactive plot with Plotly
fig = px.bar(
    x=top_trades.values / 1e9,
    y=top_trades.index,
    orientation='h',
    title="Top 15 SOFR Trades by Notional Volume",
    labels={"x": "Notional Volume ($ Billions)", "y": "Trade Label"},
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# Daily volume time series
daily_volume = trades_df.groupby(trades_df["execution_timestamp"].dt.date)["notional"].sum()

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(daily_volume.index, daily_volume.values / 1e9, alpha=0.7)
ax.set_xlabel("Date")
ax.set_ylabel("Daily Notional Volume ($ Billions)")
ax.set_title("Daily SOFR Trading Volume")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Package Analysis (Curves, Flies, Straddles)

Analyze detected package structures.

In [ ]:
# Curve trades analysis
curve_trades = trades_df[trades_df["package_type"] == "CURVE"]
print(f"\n=== CURVE TRADES ===")
print(f"Total curve trade legs: {len(curve_trades):,}")
print(f"Unique curve packages: {curve_trades['package_id'].nunique()}")

if not curve_trades.empty:
    print("\nCurve trades by label:")
    print(curve_trades.groupby("trade_label")["notional"].agg(["sum", "count"]).sort_values("sum", ascending=False).head(15))

In [ ]:
# Fly trades analysis
fly_trades = trades_df[trades_df["package_type"] == "FLY"]
print(f"\n=== BUTTERFLY TRADES ===")
print(f"Total fly trade legs: {len(fly_trades):,}")
print(f"Unique fly packages: {fly_trades['package_id'].nunique()}")

if not fly_trades.empty:
    print("\nFly trades by label:")
    print(fly_trades.groupby("trade_label")["notional"].agg(["sum", "count"]).sort_values("sum", ascending=False).head(15))

In [ ]:
# Swaption package analysis
straddle_trades = trades_df[trades_df["package_type"] == "STRADDLE"]
strangle_trades = trades_df[trades_df["package_type"] == "STRANGLE"]

print(f"\n=== SWAPTION PACKAGES ===")
print(f"Straddle legs: {len(straddle_trades):,}")
print(f"Strangle legs: {len(strangle_trades):,}")

if not straddle_trades.empty:
    print("\nStraddle trades by label:")
    print(straddle_trades.groupby("trade_label")["notional"].agg(["sum", "count"]).sort_values("sum", ascending=False).head(10))

In [ ]:
# Package type distribution
package_counts = trades_df["package_type"].value_counts()

fig = px.pie(
    values=package_counts.values,
    names=package_counts.index,
    title="Distribution of Trade Package Types",
)
fig.show()

## 5. Month-End Seasonality Analysis

What trades happen specifically at month end?

In [ ]:
# Month-end analysis
print("\n=== MONTH-END SEASONALITY ===")
month_end_analysis = report.get("month_end_analysis", pd.DataFrame())

if not month_end_analysis.empty:
    # Show trades that increase at month end
    increased = month_end_analysis[month_end_analysis["volume_ratio"] > 1.2].sort_values("volume_ratio", ascending=False)
    print("\nTrades with >20% higher volume at month-end:")
    display(increased.head(20))
else:
    print("No month-end analysis available")

In [ ]:
# Month-end vs non-month-end comparison
month_end_trades = trades_df[trades_df["is_month_end_window"] == True]
non_month_end_trades = trades_df[trades_df["is_month_end_window"] == False]

print(f"Month-end window trades: {len(month_end_trades):,}")
print(f"Non-month-end trades: {len(non_month_end_trades):,}")

print("\nTop trades during month-end window:")
me_top = month_end_trades.groupby("trade_label")["notional"].sum().sort_values(ascending=False).head(15)
print(me_top.to_string())

In [ ]:
# Visualize month-end patterns
if not month_end_analysis.empty and len(month_end_analysis) > 0:
    top_me = month_end_analysis.head(15)
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name='Month-End Avg Daily',
        x=top_me['trade_label'],
        y=top_me['event_avg_daily'] / 1e9,
    ))
    fig.add_trace(go.Bar(
        name='Non-Month-End Avg Daily',
        x=top_me['trade_label'],
        y=top_me['non_event_avg_daily'] / 1e9,
    ))
    
    fig.update_layout(
        title="Month-End vs Non-Month-End: Average Daily Volume",
        xaxis_title="Trade Label",
        yaxis_title="Avg Daily Volume ($ Billions)",
        barmode='group'
    )
    fig.show()

## 6. Quarter-End Seasonality Analysis

Quarter-end often sees specific hedging and rebalancing flows.

In [ ]:
# Quarter-end analysis
print("\n=== QUARTER-END SEASONALITY ===")
quarter_end_analysis = report.get("quarter_end_analysis", pd.DataFrame())

if not quarter_end_analysis.empty:
    # Show trades that increase at quarter end
    increased = quarter_end_analysis[quarter_end_analysis["volume_ratio"] > 1.2].sort_values("volume_ratio", ascending=False)
    print("\nTrades with >20% higher volume at quarter-end:")
    display(increased.head(20))
else:
    print("No quarter-end analysis available")

In [ ]:
# Quarter-end trades breakdown
qe_trades = trades_df[trades_df["is_quarter_end_window"] == True]

print(f"Quarter-end window trades: {len(qe_trades):,}")
print(f"Total notional: ${qe_trades['notional'].sum() / 1e9:.1f} billion")

print("\nTop trades during quarter-end window:")
qe_top = qe_trades.groupby("trade_label")["notional"].sum().sort_values(ascending=False).head(15)
print(qe_top.to_string())

## 7. FOMC Seasonality Analysis

FOMC meetings drive significant trading activity, especially in shorter tenors.

In [ ]:
# FOMC analysis
print("\n=== FOMC SEASONALITY ===")
fomc_analysis = report.get("fomc_analysis", pd.DataFrame())

if not fomc_analysis.empty:
    # Show trades that increase around FOMC
    increased = fomc_analysis[fomc_analysis["volume_ratio"] > 1.2].sort_values("volume_ratio", ascending=False)
    print("\nTrades with >20% higher volume around FOMC:")
    display(increased.head(20))
else:
    print("No FOMC analysis available")

In [ ]:
# FOMC day specific trades
fomc_day_trades = trades_df[trades_df["is_fomc_day"] == True]

print(f"\nTrades on FOMC days: {len(fomc_day_trades):,}")
print(f"Total notional: ${fomc_day_trades['notional'].sum() / 1e9:.1f} billion")

print("\nTop trades on FOMC days:")
fomc_top = fomc_day_trades.groupby("trade_label")["notional"].sum().sort_values(ascending=False).head(15)
print(fomc_top.to_string())

In [ ]:
# FOMC window trades by tenor
fomc_window = trades_df[trades_df["is_fomc_window"] == True]

print("\nFOMC window trades by tenor label:")
fomc_by_tenor = fomc_window.groupby("tenor_label")["notional"].agg(["sum", "count"]).sort_values("sum", ascending=False)
print(fomc_by_tenor.head(15))

In [ ]:
# Visualize FOMC patterns
if not fomc_analysis.empty and len(fomc_analysis) > 0:
    top_fomc = fomc_analysis.head(15)
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name='FOMC Window Avg Daily',
        x=top_fomc['trade_label'],
        y=top_fomc['event_avg_daily'] / 1e9,
    ))
    fig.add_trace(go.Bar(
        name='Non-FOMC Avg Daily',
        x=top_fomc['trade_label'],
        y=top_fomc['non_event_avg_daily'] / 1e9,
    ))
    
    fig.update_layout(
        title="FOMC Window vs Non-FOMC: Average Daily Volume",
        xaxis_title="Trade Label",
        yaxis_title="Avg Daily Volume ($ Billions)",
        barmode='group'
    )
    fig.show()

## 8. Swaption Analysis

Analyze swaption trading patterns.

In [ ]:
# Swaptions overview
swaptions = trades_df[trades_df["product_type"].str.contains("SWAPTION", na=False)]

print(f"\n=== SWAPTION ANALYSIS ===")
print(f"Total swaption trades: {len(swaptions):,}")
print(f"Total notional: ${swaptions['notional'].sum() / 1e9:.1f} billion")

print("\nBy product type:")
print(swaptions["product_type"].value_counts())

print("\nTop swaption trades by label:")
swaption_top = swaptions.groupby("trade_label")["notional"].sum().sort_values(ascending=False).head(15)
print(swaption_top.to_string())

In [ ]:
# Swaption by forward start and underlying tenor
if not swaptions.empty:
    swaption_matrix = swaptions.groupby(["forward_label", "tenor_label"])["notional"].sum().unstack(fill_value=0)
    print("\nSwaption volume matrix (forward x tenor):")
    print((swaption_matrix / 1e9).round(1))

## 9. Detailed Trade Breakdown by Tenor

Analyze trading patterns for specific benchmark tenors.

In [ ]:
# Benchmark tenors analysis
benchmark_tenors = ["2Y", "5Y", "10Y", "30Y"]

print("\n=== BENCHMARK TENOR ANALYSIS ===")
for tenor in benchmark_tenors:
    tenor_trades = trades_df[trades_df["tenor_label"] == tenor]
    if len(tenor_trades) > 0:
        print(f"\n--- {tenor} Tenor ---")
        print(f"Total trades: {len(tenor_trades):,}")
        print(f"Total notional: ${tenor_trades['notional'].sum() / 1e9:.1f}B")
        
        # Breakdown by forward start
        by_forward = tenor_trades.groupby("forward_label")["notional"].sum().sort_values(ascending=False)
        print(f"By forward start:")
        for fwd, vol in by_forward.head(5).items():
            print(f"  {fwd}: ${vol/1e9:.1f}B")

In [ ]:
# Common curve trades
print("\n=== COMMON CURVE TRADE STRUCTURES ===")

curve_structures = [
    ("2Y", "5Y", "2s5s"),
    ("2Y", "10Y", "2s10s"),
    ("5Y", "10Y", "5s10s"),
    ("5Y", "30Y", "5s30s"),
    ("10Y", "30Y", "10s30s"),
]

for short_tenor, long_tenor, name in curve_structures:
    # Find trades that could be legs of this curve
    short_trades = trades_df[trades_df["tenor_label"] == short_tenor]
    long_trades = trades_df[trades_df["tenor_label"] == long_tenor]
    
    if len(short_trades) > 0 and len(long_trades) > 0:
        combined_vol = short_trades["notional"].sum() + long_trades["notional"].sum()
        print(f"{name}: ${combined_vol/1e9:.1f}B combined volume ({len(short_trades)} + {len(long_trades)} trades)")

## 10. Summary Report

Generate a comprehensive summary of the seasonality analysis.

In [ ]:
print("="*80)
print("SOFR SDR SEASONALITY ANALYSIS - SUMMARY REPORT")
print("="*80)

print(f"\nAnalysis Period: {start_date} to {end_date}")
print(f"Total Trades Analyzed: {len(trades_df):,}")
print(f"Total Notional: ${trades_df['notional'].sum() / 1e12:.2f} trillion")

print("\n" + "="*40)
print("TOP 10 TRADED STRUCTURES")
print("="*40)
top_10 = trades_df.groupby("trade_label")["notional"].sum().sort_values(ascending=False).head(10)
for i, (label, vol) in enumerate(top_10.items(), 1):
    print(f"{i:2d}. {label:15s} ${vol/1e9:8.1f}B")

print("\n" + "="*40)
print("MONTH-END PATTERNS")
print("="*40)
if not month_end_analysis.empty:
    high_me = month_end_analysis[month_end_analysis["volume_ratio"] > 1.5].head(5)
    print("Trades with >50% higher volume at month-end:")
    for _, row in high_me.iterrows():
        print(f"  - {row['trade_label']}: {row['volume_ratio']:.1f}x normal volume")

print("\n" + "="*40)
print("QUARTER-END PATTERNS")
print("="*40)
if not quarter_end_analysis.empty:
    high_qe = quarter_end_analysis[quarter_end_analysis["volume_ratio"] > 1.5].head(5)
    print("Trades with >50% higher volume at quarter-end:")
    for _, row in high_qe.iterrows():
        print(f"  - {row['trade_label']}: {row['volume_ratio']:.1f}x normal volume")

print("\n" + "="*40)
print("FOMC PATTERNS")
print("="*40)
if not fomc_analysis.empty:
    high_fomc = fomc_analysis[fomc_analysis["volume_ratio"] > 1.5].head(5)
    print("Trades with >50% higher volume around FOMC:")
    for _, row in high_fomc.iterrows():
        print(f"  - {row['trade_label']}: {row['volume_ratio']:.1f}x normal volume")

print("\n" + "="*40)
print("PACKAGE TRADE SUMMARY")
print("="*40)
print(f"Curve trades detected: {trades_df[trades_df['package_type']=='CURVE']['package_id'].nunique()}")
print(f"Fly trades detected: {trades_df[trades_df['package_type']=='FLY']['package_id'].nunique()}")
print(f"Straddles detected: {trades_df[trades_df['package_type']=='STRADDLE']['package_id'].nunique()}")
print(f"Strangles detected: {trades_df[trades_df['package_type']=='STRANGLE']['package_id'].nunique()}")

print("\n" + "="*80)

In [ ]:
# Export results to CSV if needed
# trades_df.to_csv("sofr_seasonality_trades.csv", index=False)
# print("Results exported to sofr_seasonality_trades.csv")

## 11. Extended Analysis: Days from Event

Analyze how volume changes as we approach month-end, quarter-end, and FOMC.

In [ ]:
# Volume by days to month end
me_window = trades_df[trades_df["days_to_month_end"].notna()]

if not me_window.empty:
    vol_by_days = me_window.groupby("days_to_month_end")["notional"].sum()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(vol_by_days.index, vol_by_days.values / 1e9)
    ax.set_xlabel("Days Until Month End")
    ax.set_ylabel("Total Volume ($ Billions)")
    ax.set_title("Trading Volume by Days Until Month End")
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Volume by days to FOMC
fomc_window = trades_df[trades_df["days_to_fomc"].notna()]

if not fomc_window.empty:
    vol_by_days = fomc_window.groupby("days_to_fomc")["notional"].sum()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(vol_by_days.index, vol_by_days.values / 1e9, color='green')
    ax.set_xlabel("Days Until FOMC")
    ax.set_ylabel("Total Volume ($ Billions)")
    ax.set_title("Trading Volume by Days Until FOMC")
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()

## 12. Heatmap: Trade Volume by Tenor and Forward

Visualize the distribution of trades across the tenor/forward matrix.

In [ ]:
# Create tenor x forward heatmap for OIS swaps
ois_swaps = trades_df[trades_df["product_type"] == "OIS_SWAP"]

if not ois_swaps.empty:
    # Create pivot table
    heatmap_data = ois_swaps.groupby(["forward_label", "tenor_label"])["notional"].sum().unstack(fill_value=0)
    
    # Order columns by tenor
    tenor_order = ["1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y", "12Y", "15Y", "20Y", "25Y", "30Y"]
    existing_tenors = [t for t in tenor_order if t in heatmap_data.columns]
    heatmap_data = heatmap_data[existing_tenors]
    
    # Plot heatmap
    fig = px.imshow(
        heatmap_data.values / 1e9,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        labels={"x": "Underlying Tenor", "y": "Forward Start", "color": "Volume ($B)"},
        title="OIS Swap Volume: Forward Start vs Underlying Tenor",
        color_continuous_scale="Blues",
    )
    fig.update_layout(width=1000, height=600)
    fig.show()

In [ ]:
# Print the heatmap data
print("\nVolume Matrix ($ Billions):")
print((heatmap_data / 1e9).round(1))